# 09b · Second external test, with the SS gaps filled (sensitivity)

**Why this notebook exists.** In `External_BHs_SR_Elev.csv` only the ST (Shelby tube) samples have a USCS symbol; the SS (SPT) samples between them were never classified in the logs. Under the training rule (data.fill_gaps, condition c) such an unclassified record blocks gap filling, so notebook 09 scored only the 84 ST cells. After seeing that result, the student decided, as a geotechnical judgement, that an unclassified SS sample between two samples of the same class belongs to that class.

| Item | Setting |
|---|---|
| Change | `fill_gaps(..., unlabelled_blocks=False)`: a record without a USCS symbol no longer blocks a fill. Everything else is the same as notebook 09 (same class above and below, at most 3 m, nothing above the first or below the last measured cell) |
| Model | the same frozen model and threshold; nothing is refitted or tuned |
| Status | a **sensitivity analysis decided after notebook 09 was seen**. Notebook 09 stays the primary independent test; report both. The training data used the stricter rule (on the main Downtown data the relaxed rule would add 157 filled cells to the 917) |
| Cells | **all cells** (84 measured + filled) and **measured only** (identical to notebook 09) |


In [ ]:
import os, sys, json
from pathlib import Path

# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner())

import joblib
from siemreap import data, geo, metrics as M, nested_cv as NCV, plots
from siemreap.models import DepthPrior
AGG = config.stage_dir(cfg, "09_new_external", "aggregate")
RES = config.stage_dir(cfg, "09_new_external", "restricted")
mt = pd.read_csv(config.processed_dir(cfg) / "cells_all.csv")
frozen = json.loads((config.stage_dir(cfg, "05_final", "aggregate") / "final_model.json").read_text())
model_path = config.stage_dir(cfg, "05_final", "restricted") / frozen["model_file"]
model_sha = config.sha256_file(model_path)
assert model_sha == frozen["model_sha256"], "the model file differs from the frozen record"
model = joblib.load(model_path)
print("frozen model:", frozen["model"], frozen["block_m"], "m | threshold", frozen["threshold"], "| SHA-256", model_sha[:16])

## 1. The new boreholes, SS gaps filled


In [ ]:
new_raw, new_info = data.load_new_external(cfg)
print(json.dumps(new_info, indent=1))
cells, dinfo = data.discretise(new_raw, cfg)
cells, finfo = data.fill_gaps(cells, new_raw, cfg, unlabelled_blocks=False)
assert np.allclose(cells.Top_Elevation_m, cells.Ground_Elevation_m - cells.depth_top_m)
train_xy = mt[mt.in_A].drop_duplicates("BH_Code")[["UTM_Easting_m", "UTM_Northing_m"]].values
bh = cells.drop_duplicates("BH_Code").set_index("BH_Code")[["UTM_Easting_m", "UTM_Northing_m"]]
bh["dist_nearest_train_m"] = geo.nearest_distance(bh.values, train_xy).round(0)
bh = bh.join(cells.groupby("BH_Code").agg(cells=("measured", "size"), measured=("measured", "sum")))
display(bh[["cells", "measured", "dist_nearest_train_m"]])    # counts only: no class values before the evaluation
print(json.dumps({k: finfo[k] for k in ("gaps", "different_class", "too_long", "touched_by_a_record", "filled_cells")}))
print({"cells": len(cells), "measured": int(cells.measured.sum()), "filled": int((cells.measured == 0).sum()),
       "boreholes": cells.BH_Code.nunique()})

## 2. Evaluation


In [ ]:
ALLOW_REEVALUATION = False      # set True only with a documented reason; report it in the thesis
record_path = AGG / "new_external_evaluation_ssfill.json"
BANDS = [0, 300, 700, 1000, np.inf]
band_lab = ["< 300 m", "300-700 m", "700-1000 m", "> 1000 m"]
if MODE == "smoke":
    rng = np.random.default_rng(1)
    stand_in = rng.choice(mt[mt.in_A].BH_Code.drop_duplicates().sort_values().values, 8, replace=False)
    ev = mt[mt.BH_Code.isin(stand_in)].copy()
    label = "SMOKE STAND-IN (development boreholes; the new boreholes are not scored)"
else:
    if record_path.exists():
        prev = json.loads(record_path.read_text())
        if prev["model_sha256"] != model_sha and not ALLOW_REEVALUATION:
            raise RuntimeError("These boreholes were already evaluated with a different model; set ALLOW_REEVALUATION "
                               "only with a documented reason.")
    ev = cells.copy()
    label = "NEW EXTERNAL BOREHOLES, SS GAPS FILLED (sensitivity)"
ev["dist_nearest_train_m"] = geo.nearest_distance(ev[["UTM_Easting_m", "UTM_Northing_m"]].values, train_xy)
ev["group"] = pd.cut(ev.dist_nearest_train_m, BANDS, right=False, labels=band_lab).astype(str)
p = model.predict_proba(ev[["UTM_Easting_m", "UTM_Northing_m", "Top_Elevation_m"]].values)[:, 1]
ev_m = ev.assign(p_clay=p, y_pred=(p >= frozen["threshold"]).astype(int))
train_A = mt[mt.in_A]
ref = DepthPrior(bottom_m=cfg["depth"]["bottom_m"]).fit(train_A.depth_top_m.values, train_A.y.values)
ref_thr = NCV.fit_final(mt, cfg, frozen["scenario"], "DEPTH_PRIOR", int(frozen["block_m"]), MODE)["threshold"]
p_ref = ref.predict_proba(ev.depth_top_m.values)[:, 1]
ev_r = ev.assign(p_clay=p_ref, y_pred=(p_ref >= ref_thr).astype(int))
rows = []
for name, e in ((frozen["model"], ev_m), ("DEPTH_PRIOR", ev_r)):
    for subset, d0 in (("all cells", e), ("measured only", e[e.measured == 1])):
        rows.append({"model": name, "cells": subset, "group": "all", **M.metric_row(d0.y, d0.p_clay, d0.y_pred)})
        for g, d in d0.groupby("group"):
            rows.append({"model": name, "cells": subset, "group": g, **M.metric_row(d.y, d.p_clay, d.y_pred)})
res = pd.DataFrame(rows)
print(label)
display(res[["model", "cells", "group", "n", "accuracy", "balanced_accuracy", "macro_f1", "recall_clay",
             "recall_sand", "roc_auc", "brier"]].round(3))
record_path.write_text(json.dumps({"label": label, "model_sha256": model_sha, "data": new_info, "metrics": res.to_dict("records"),
                                   "depth_prior_threshold": ref_thr, "fill_rule": "unlabelled_blocks=False", "fill_info": finfo, "mode": MODE}, indent=1, default=str))
ev_m.to_csv(RES / "new_external_predictions_ssfill.csv", index=False)

## 3. By borehole, and the comparison figure

In [ ]:
per = []
for name, e in ((frozen["model"], ev_m), ("DEPTH_PRIOR", ev_r)):
    for code, g in e.groupby("BH_Code"):
        per.append({"model": name, "BH_Code": code, "n": len(g), "clay_share": g.y.mean(), "accuracy": (g.y == g.y_pred).mean(),
                    "dist_nearest_train_m": round(g.dist_nearest_train_m.iloc[0])})
per = pd.DataFrame(per).pivot_table(index=["BH_Code", "dist_nearest_train_m", "n", "clay_share"], columns="model",
                                    values="accuracy").round(3).reset_index().sort_values("dist_nearest_train_m")
display(per); per.to_csv(RES / "new_external_by_borehole_ssfill.csv", index=False)
groups = ["all"] + [b for b in band_lab if b in set(res.group)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), dpi=150, sharey=True)
for ax, k, lab in zip(axes, ["macro_f1", "roc_auc"], ["Macro-F1", "ROC-AUC"]):
    for j, (name, col) in enumerate([(frozen["model"], "#A0004E"), ("DEPTH_PRIOR", "#9E9E9E")]):
        d = res[(res.model == name) & (res.cells == "all cells")].set_index("group").reindex(groups)
        ax.bar(np.arange(len(groups)) + (j - 0.5) * 0.38, d[k], width=0.36, color=col,
               label=name.replace("DEPTH_PRIOR", "Depth only"))
        for x, v in zip(np.arange(len(groups)) + (j - 0.5) * 0.38, d[k]):
            if pd.notna(v):
                ax.text(x, v + 0.01, f"{v:.2f}", ha="center", fontsize=7)
    n_ = res[(res.model == frozen["model"]) & (res.cells == "all cells")].set_index("group").reindex(groups).n
    ax.set_xticks(range(len(groups)), [f"{g}\nn = {int(n)}" if pd.notna(n) else g for g, n in zip(groups, n_)], fontsize=8)
    ax.axhline(0.5, color="black", linestyle=":", linewidth=0.8)
    ax.set_ylim(0, 1); ax.set_title(f"New external boreholes, SS gaps filled: {lab}", loc="left", fontsize=10)
    ax.set_xlabel("Distance to the nearest training borehole")
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(AGG / "fig_new_external_ssfill.png", bbox_inches="tight")
fig = plots.profiles(ev_m.assign(outer_fold=0, threshold=frozen["threshold"]), ev_m.BH_Code.drop_duplicates(),
                     title=f"{label}: observed vs predicted ({frozen['model']})")
fig.savefig(RES / "fig_new_external_profiles_ssfill.png", bbox_inches="tight")

## 4. Side by side with notebook 09 (all boreholes)


In [ ]:
first = AGG / "new_external_evaluation.json"
if first.exists():
    a = pd.DataFrame(json.loads(first.read_text())["metrics"]).assign(run="09: ST cells only")
    both = pd.concat([a, res.assign(run="09b: SS gaps filled")])
    both = both[(both.group == "all")]
    display(both[["run", "model", "cells", "n", "accuracy", "balanced_accuracy", "macro_f1", "roc_auc", "brier"]].round(3))
else:
    print("run notebook 09 first")


**For the thesis.** Notebook 09 is the primary independent test. This notebook is a sensitivity analysis whose fill rule was decided after notebook 09 was seen; state that, and report both. The model and threshold are unchanged, so any difference comes only from which cells are scored.